## Assignment 16

In this notebook we are going to extract information from resume images about the candidate. 

## Downloading and understanding dataset (Resume Dataset)

Resume Dataset containes the PDFs of resumes reguarding diffrent job roles.

### Getting the Dataset

In [1]:
import kagglehub

# 1. Download the dataset (returns the local path to the folder)
dataset_path = kagglehub.dataset_download("snehaanbhawal/resume-dataset")

print(f"Dataset downloaded to: {dataset_path}")

c:\Users\Nabeel-IT\miniconda3\envs\genai_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset downloaded to: C:\Users\Nabeel-IT\.cache\kagglehub\datasets\snehaanbhawal\resume-dataset\versions\1


In [2]:
import pandas as pd
import os
result_resume_csv_path = os.path.join(dataset_path, "Resume", "Resume.csv")
result_resume_csv = pd.read_csv(result_resume_csv_path)

print(result_resume_csv.head())

         ID                                         Resume_str  \
0  16852973           HR ADMINISTRATOR/MARKETING ASSOCIATE\...   
1  22323967           HR SPECIALIST, US HR OPERATIONS      ...   
2  33176873           HR DIRECTOR       Summary      Over 2...   
3  27018550           HR SPECIALIST       Summary    Dedica...   
4  17812897           HR MANAGER         Skill Highlights  ...   

                                         Resume_html Category  
0  <div class="fontsize fontface vmargins hmargin...       HR  
1  <div class="fontsize fontface vmargins hmargin...       HR  
2  <div class="fontsize fontface vmargins hmargin...       HR  
3  <div class="fontsize fontface vmargins hmargin...       HR  
4  <div class="fontsize fontface vmargins hmargin...       HR  


### Resume Dataset Overview

This dataset contains **2,400+ resumes** provided in two formats: raw text strings and original PDFs. It is structured to support tasks like Natural Language Processing (NLP), classification, and document analysis.

---

#### 📂 Directory Structure
The files are organized systematically within the `data` directory. Each professional category acts as a parent folder containing the individual PDF resumes.

*   **Format 1:** Text strings (for direct programmatic processing).
*   **Format 2:** PDF files (stored as `./data/[Category]/[filename].pdf`).

---

#### 🏷️ Present Categories
The dataset is classified into the following **24 professional domains**:

| | | | |
| :--- | :--- | :--- | :--- |
| 📁 Accountant | 📁 Advocate | 📁 Agriculture | 📁 Apparel |
| 📁 Arts | 📁 Automobile | 📁 Aviation | 📁 Banking |
| 📁 BPO | 📁 Business-Development | 📁 Chef | 📁 Construction |
| 📁 Consultant | 📁 Designer | 📁 Digital-Media | 📁 Engineering |
| 📁 Finance | 📁 Fitness | 📁 Healthcare | 📁 HR |
| 📁 Information-Technology | 📁 Public-Relations | 📁 Sales | 📁 Teacher |

---

> **Tip:** When loading this data, ensure your script iterates through the sub-folders in the `data` directory to correctly map the filenames to their respective labels.

## 1. Document Ingestion & OCR
The initial phase involves converting various file formats (**PDF**, **DocX**, or **Images**) into machine-readable text while attempting to preserve the original document layout.

#### 📚 Recommended Libraries
*   **For Digital PDFs:** [PyMuPDF](https://pymupdf.readthedocs.io/) or [pdfplumber](https://github.com/jsvine/pdfplumber) (best for extracting text with coordinates).
*   **For Scanned Documents:** [Tesseract OCR](https://github.com/tesseract-ocr/tesseract) or [Amazon Textract](https://aws.amazon.com/textract/) (handles handwriting and complex forms).

---

### ⚠️ The Layout Challenge
Standard OCR engines often process text in a strictly linear fashion (left-to-right). This creates significant issues for **multi-column resumes**, as the logic often:
1.  Reads across the entire page width.
2.  Scrambles data from different sections (e.g., merging "Experience" with "Contact Info").

**Solution:** Implement **Layout Analysis** to detect bounding boxes. By identifying columns as separate blocks, you can extract text in the correct reading order.

> **Pro Tip:** Use layout-aware tools like `LayoutParser` or deep learning models (like Detectron2) to segment the page before performing OCR.

### Text-Extraction: pymupdf4llm (an official extension of PyMuPDF)

The pymupdf4llm library is an official extension of PyMuPDF specifically built for data extraction. It is used because it converts PDFs directly into Markdown format. This conversion is essential for Intelligent Document Processing because Large Language Models (LLMs), such as Gemini, understand Markdown significantly better than raw, unstructured text. By preserving structural elements like header hierarchies (e.g., "## Experience") and lists, pymupdf4llm allows the LLM to accurately interpret the layout and context of complex documents like resumes.

In [3]:
import os
import time
import json
import csv
from pathlib import Path
import pymupdf4llm
from pydantic import BaseModel, Field
from typing import List, Optional


In [4]:
class WorkExperience(BaseModel):
    job_title: Optional[str] = Field(description="The exact job title held by the candidate.")
    company_name: Optional[str] = Field(description="The name of the company or employer.")
    dates_of_employment: Optional[str] = Field(description="The start and end dates of the job.")
    responsibilities: List[str] = Field(default_factory=list, description="A list of specific duties and achievements.")

class Education(BaseModel):
    degree: Optional[str] = Field(description="The name of the degree, e.g., Bachelor of Science.")
    institution: Optional[str] = Field(description="The name of the university or school.")
    graduation_year: Optional[str] = Field(description="The year the degree was completed.")

class ResumeData(BaseModel):
    name: str                          
    email: str                         
    phone: str                         
    location: Optional[str]            
    summary: Optional[str]             
    total_years_experience: str        
    skills: List[str] = Field(default_factory=list)                 
    certifications: List[str] = Field(default_factory=list, description="List of professional certifications and licenses.")
    experience: List[WorkExperience] = Field(default_factory=list, description="An array of all past work experience and jobs.")
    education: List[Education] = Field(default_factory=list, description="An array of all educational degrees.")

In [5]:
#Gemini API Key
from dotenv import load_dotenv
from google import genai

# 1. Load the hidden variables from your .env file
load_dotenv()

# 2. Initialize the client securely using the api_key parameter
client = genai.Client(api_key=os.getenv("GENAI_API_KEY"))

In [6]:
# Define paths
ROOT_FOLDER = "C:\\Users\\Nabeel-IT\\.cache\\kagglehub\\datasets\\snehaanbhawal\\resume-dataset\\versions\\1\\data\\data" # Your main folder containing the subfolders
CSV_FILENAME = "extracted_resumes.csv"

In [7]:
# 3. Setup the CSV File and write the header row
# We open in 'w' mode first just to create the file and headers
with open(CSV_FILENAME, mode= 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow([
        "Profession",           # From Folder
        "File_Name",            # From File
        "Name",                 # 1
        "Email",                # 2
        "Phone",                # 3
        "Location",             # 4
        "Summary",              # 5
        "Total_Years_Experience", # 6
        "Skills",               # 7
        "Certifications",       # 8
        "Degrees",              # 9 (Extracted from Education list)
        "Institutions",         # 10 (Extracted from Education list)
        "Graduation_Years",     # 11 (Extracted from Education list)
        "Job_Titles",           # 12 (Extracted from WorkExperience list)
        "Companies",            # 13 (Extracted from WorkExperience list)
        "Employment_Dates",     # 14 (Extracted from WorkExperience list)
        "Responsibilities",     # 15 (Extracted from WorkExperience list)
        "Status"                # Success/Error Logging
    ])

In [8]:
# 4. Find all PDFs in all subfolders
# rglob("*.pdf") recursively searches all subfolders!
pdf_files = list(Path(ROOT_FOLDER).rglob("*.pdf"))
print(f"Found {len(pdf_files)} resumes to process.")

Found 2484 resumes to process.


In [9]:
# 5. Process each PDF and append to CSV immediately
# We open the CSV in 'a' (append) mode. If the script crashes at resume #250, 
# you won't lose the first 249!
with open(CSV_FILENAME, mode='a', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)

    for pdf_path in pdf_files:
        # Extract metadata from the file path
        file_name = pdf_path.name
        profession = pdf_path.parent.name # This gets the subfolder name!
        
        print(f"Processing: {file_name} (Folder: {profession})...")
        
        try:
            # A. Extract Markdown using PyMuPDF
            md_text = pymupdf4llm.to_markdown(str(pdf_path))
            
            # B. Send to Gemini
            prompt = f"""
                You are an expert AI resume parser and data extraction assistant. Your task is to extract highly accurate, structured information from the provided applicant resume. 

                The resume text has been extracted from a PDF and converted into Markdown format.

                ### EXTRACTION RULES:
                1. **Accuracy First:** Extract information exactly as it appears in the text. Do not hallucinate, guess, or infer information that is not explicitly stated.
                2. **Missing Data:** If a specific piece of information (e.g., Phone Number or Location) is not present in the resume, leave the field null or empty. Do not use placeholders like "N/A" or "Unknown".
                3. **Summary:** If there is no explicit summary paragraph, extract a 1-2 sentence overview based on their most recent experience.
                4. **Skills:** Extract all technical, soft, and industry-specific skills. Normalize them where obvious (e.g., "Amazon Web Services" to "AWS").
                5. **Total Years Experience:** Calculate the total years of professional experience based on the employment dates provided. Round to the nearest whole number (e.g., "5").
                6. **Work Experience (Nested):** Extract every distinct role. Group the Job Title, Company, Dates, and a list of specific Responsibilities for each role. 
                7. **Education (Nested):** Extract every degree or certification. Group the Degree Name, Institution, and Graduation Year.
                8. **Certifications:** List any active licenses or professional certifications separately from standard education.

                ### RESUME MARKDOWN TEXT:
                {md_text}
            """
            response = client.models.generate_content(
                model='gemini-2.5-flash', 
                contents=prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': ResumeData,
                    'temperature': 0.1 # Low temperature for factual extraction
                },
            )
            
            # C. Parse the JSON response
            extracted_data = json.loads(response.text)

            print(extracted_data)
            
            # D. Flatten Lists and Nested Objects safely (Dictionary version)
            skills_str = ", ".join(extracted_data.get("skills") or [])
            certs_str = ", ".join(extracted_data.get("certifications") or [])
            
            degrees_str = ", ".join([edu.get("degree") for edu in (extracted_data.get("education") or []) if edu.get("degree")])
            institutions_str = ", ".join([edu.get("institution") for edu in (extracted_data.get("education") or []) if edu.get("institution")])
            grad_years_str = ", ".join([str(edu.get("graduation_year")) for edu in (extracted_data.get("education") or []) if edu.get("graduation_year")])
            
            job_titles_str = ", ".join([job.get("job_title") for job in (extracted_data.get("experience") or []) if job.get("job_title")])
            companies_str = ", ".join([job.get("company_name") for job in (extracted_data.get("experience") or []) if job.get("company_name")])
            dates_str = ", ".join([job.get("dates_of_employment") for job in (extracted_data.get("experience") or []) if job.get("dates_of_employment")])
            
            # Join responsibilities with a semi-colon to differentiate from the commas inside the bullet points
            resp_str = " ; ".join([" ".join(job.get("responsibilities") or []) for job in (extracted_data.get("experience") or []) if job.get("responsibilities")])
            

            # E. Write the row to the CSV
            writer.writerow([
                profession,
                file_name,
                extracted_data.get("name", "N/A"),
                extracted_data.get("email", "N/A"),
                extracted_data.get("phone", "N/A"),
                extracted_data.get("location", "N/A"),
                extracted_data.get("summary", "N/A"),
                extracted_data.get("total_years_experience", "N/A"),
                skills_str,
                certs_str,
                degrees_str,
                institutions_str,
                grad_years_str,
                job_titles_str,
                companies_str,
                dates_str,
                resp_str,
                "Success"
            ])
            
        except Exception as e:
            # If a PDF is corrupted or Gemini fails, log the error but DON'T crash the loop
            print(f"  -> Error on {file_name}: {e}")
            writer.writerow([profession, file_name, "", "", "", "", "", f"Error: {e}"])
        
        # F. VERY IMPORTANT: Pause to avoid hitting API Rate Limits!
        # Free tier Gemini APIs have Requests Per Minute (RPM) limits.
        time.sleep(7) 

print("Batch processing complete!")

Processing: 10554236.pdf (Folder: ACCOUNTANT)...
{'name': 'null', 'email': 'null', 'phone': 'null', 'location': 'null', 'summary': 'Financial Accountant specializing in financial planning, reporting and analysis within the Department of Defense.', 'total_years_experience': '19', 'skills': ['Accounting', 'Financial Planning', 'Financial Reporting', 'Financial Analysis', 'Account Reconciliations', 'Accounting Operations', 'Critical Thinking', 'ERP Software', 'Facilitation', 'General Ledger', 'DEAMS', 'Procure to Pay', 'Orders to Cash', 'Budget to Report', 'System Testing', 'Budget Management', 'Defense Travel System (DTS)', 'General Accounting and Finance System (GAFS)', 'Government Purchase Card (GPC) Management', 'Performance Metrics', 'Automated Tool Development', 'Access On-Line', 'Open Document Analysis (ODA)', 'FMSuite', 'Remote Training', 'Managerial Reporting', 'System Reporting', 'Fund Reconciliation', 'Workforce Management', 'Performance Standards', 'Suspense Account Management

KeyboardInterrupt: 